---
## TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

> **Tenggat waktu:** dikumpulkan paling lambat **sebelum Pertemuan 12 dimulai**.
> **Sifat tugas:** individu.

### Konteks / Skenario

Manajemen "Kopi Nusantara" (skenario yang sama dari Pertemuan 7, 9, dan 10) ingin memiliki alat bantu **prediksi omzet harian** tiap cabang, agar dapat merencanakan stok bahan baku dengan lebih baik. Anda ditugaskan membangun model regresi untuk memprediksi `total_penjualan` sebuah pesanan berdasarkan atribut-atributnya.

### Menyiapkan Dataset

Jalankan cell berikut untuk membuat dataset.

In [1]:
import numpy as np
import pandas as pd

np.random.seed(111)
n = 2500
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas11 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "harga_satuan": np.random.choice([15000, 18000, 20000, 22000, 25000, 30000], size=n),
})
data_tugas11["total_penjualan"] = data_tugas11["qty"] * data_tugas11["harga_satuan"] + np.random.normal(0, 8000, size=n)
data_tugas11.to_csv("tugas11_penjualan.csv", index=False)

print(f"Dataset berhasil dibuat: {len(data_tugas11)} baris")
print(data_tugas11.head())

Dataset berhasil dibuat: 2500 baris
        nama_menu      cabang waktu_pesan  qty  harga_satuan  total_penjualan
0  Kentang Goreng    Semarang       Malam    1         30000     27750.163532
1  Kentang Goreng  Yogyakarta        Sore    3         20000     64216.160744
2  Kentang Goreng    Semarang       Siang    5         22000    108722.792592
3  Kentang Goreng  Yogyakarta        Pagi    2         20000     54483.205872
4      Roti Bakar    Magelang       Malam    2         30000     58157.944120


### Instruksi Pengerjaan

Buat notebook baru **`Tugas11_[NPM]_[Nama Lengkap].ipynb`**, buat `SparkSession`, baca `tugas11_penjualan.csv`, lalu kerjakan bagian **A sampai E** berikut:

---

**A. Feature Engineering** *(bobot 20%)*

Gunakan `StringIndexer` untuk mengubah kolom `nama_menu`, `cabang`, dan `waktu_pesan` menjadi indeks numerik. Gabungkan seluruh fitur (`qty`, `harga_satuan`, dan ketiga kolom hasil indexing) menjadi satu kolom `features` menggunakan `VectorAssembler`.

**B. Train/Test Split & Pelatihan Model** *(bobot 20%)*

Bagi data menjadi 75% training dan 25% testing (`seed=101`). Latih model `LinearRegression` dengan `labelCol="total_penjualan"`.

**C. Evaluasi Model** *(bobot 20%)*

Hitung dan tampilkan **RMSE** dan **R²** dari model pada data testing. Tulis 1-2 kalimat interpretasi: apakah model ini cukup baik untuk dipakai memprediksi omzet?

**D. Eksperimen Perbandingan Fitur** *(bobot 25%)*

Latih **model kedua** yang HANYA menggunakan fitur `qty` dan `harga_satuan` (tanpa informasi menu/cabang/waktu). Bandingkan R² model kedua ini dengan model utama pada bagian C. Tulis kesimpulan: apakah menambahkan informasi menu/cabang/waktu benar-benar meningkatkan akurasi prediksi secara berarti?

**E. Prediksi Skenario Baru** *(bobot 15%)*

Buat sebuah DataFrame baru berisi **3 skenario pesanan (data dummy)** (boleh bebas kombinasi menu/cabang/waktu/qty/harga, namun harus melalui tahapan `StringIndexer` dan `VectorAssembler` yang sama seperti data training — perhatikan: gunakan model `StringIndexer` yang **sama** yang sudah di-`fit()` sebelumnya, jangan membuat `StringIndexer` baru). Tampilkan hasil prediksi `total_penjualan` untuk ketiga skenario tersebut.

---

In [2]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

spark = SparkSession.builder.appName("Tugas11-KopiNusantara").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

df = spark.read.csv("tugas11_penjualan.csv", header=True, inferSchema=True)
df.show(5)

26/11/18 10:25:42 WARN Utils: Your hostname, yan resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/11/18 10:25:42 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/18 10:25:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/18 10:25:45 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
                                                                                

+--------------+----------+-----------+---+------------+------------------+
|     nama_menu|    cabang|waktu_pesan|qty|harga_satuan|   total_penjualan|
+--------------+----------+-----------+---+------------+------------------+
|Kentang Goreng|  Semarang|      Malam|  1|       30000|27750.163531558544|
|Kentang Goreng|Yogyakarta|       Sore|  3|       20000| 64216.16074371864|
|Kentang Goreng|  Semarang|      Siang|  5|       22000|108722.79259154423|
|Kentang Goreng|Yogyakarta|       Pagi|  2|       20000|54483.205871792496|
|    Roti Bakar|  Magelang|      Malam|  2|       30000| 58157.94412015849|
+--------------+----------+-----------+---+------------+------------------+
only showing top 5 rows



In [3]:
# A. Feature Engineering
indexer_menu = StringIndexer(inputCol="nama_menu", outputCol="nama_menu_idx").fit(df)
indexer_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx").fit(df)
indexer_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_pesan_idx").fit(df)

df_indexed = indexer_menu.transform(df)
df_indexed = indexer_cabang.transform(df_indexed)
df_indexed = indexer_waktu.transform(df_indexed)

assembler = VectorAssembler(
    inputCols=["qty", "harga_satuan", "nama_menu_idx", "cabang_idx", "waktu_pesan_idx"],
    outputCol="features"
)
df_final = assembler.transform(df_indexed)
df_final.select("features", "total_penjualan").show(5, truncate=False)

+-------------------------+------------------+
|features                 |total_penjualan   |
+-------------------------+------------------+
|[1.0,30000.0,5.0,2.0,1.0]|27750.163531558544|
|[3.0,20000.0,5.0,1.0,3.0]|64216.16074371864 |
|[5.0,22000.0,5.0,2.0,2.0]|108722.79259154423|
|[2.0,20000.0,5.0,1.0,0.0]|54483.205871792496|
|[2.0,30000.0,4.0,0.0,1.0]|58157.94412015849 |
+-------------------------+------------------+
only showing top 5 rows



In [4]:
# B. Train/Test Split & Pelatihan Model
train, test = df_final.randomSplit([0.75, 0.25], seed=101)
print(f"Jumlah data training : {train.count()}")
print(f"Jumlah data testing  : {test.count()}")

lr = LinearRegression(featuresCol="features", labelCol="total_penjualan")
model = lr.fit(train)
print("Koefisien:", model.coefficients)
print("Intercept:", model.intercept)

Jumlah data training : 1839
Jumlah data testing  : 661
Koefisien: [22046.663271025758,3.022920868160672,-123.21425598657399,279.28452461677676,339.6387953231708]
Intercept: -66903.17766485889


In [5]:
# C. Evaluasi Model
predictions = model.transform(test)

evaluator_rmse = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2")

rmse = evaluator_rmse.evaluate(predictions)
r2 = evaluator_r2.evaluate(predictions)

print(f"RMSE : Rp {rmse:,.0f}")
print(f"R²   : {r2:.4f}  ({r2*100:.1f}% variasi data berhasil dijelaskan model)")

RMSE : Rp 10,901
R²   : 0.9107  (91.1% variasi data berhasil dijelaskan model)


Model ini bisa jelasin sekitar 91% variasi total_penjualan, dengan rata-rata error sekitar Rp 10.900 per pesanan. Buat perkiraan kasar omzet dan rencana stok, ini udah cukup layak. Tapi errornya masih lebih gede dari noise acak di datanya (sekitar Rp 8.000), karena model linear nggak nangkep bahwa total penjualan itu hasil perkalian qty x harga_satuan. Jadi belum cocok kalau butuh prediksi yang presisi banget.

In [6]:
# D. Eksperimen Perbandingan Fitur
assembler_dasar = VectorAssembler(inputCols=["qty", "harga_satuan"], outputCol="features_dasar")
df_dasar = assembler_dasar.transform(df_indexed)

train_d, test_d = df_dasar.randomSplit([0.75, 0.25], seed=101)
lr_d = LinearRegression(featuresCol="features_dasar", labelCol="total_penjualan")
model_d = lr_d.fit(train_d)
pred_d = model_d.transform(test_d)

r2_dasar = evaluator_r2.evaluate(pred_d)
print(f"R² model utama (5 fitur)   : {r2:.4f}")
print(f"R² model dasar (2 fitur)   : {r2_dasar:.4f}")

R² model utama (5 fitur)   : 0.9107
R² model dasar (2 fitur)   : 0.9108


R² kedua model hampir sama (0.9107 vs 0.9108), jadi nambahin info menu, cabang, dan waktu pesan nggak bikin prediksi jadi lebih akurat secara berarti. Di data ini, total penjualan hampir sepenuhnya ditentuin sama qty dan harga_satuan, sedangkan menu, cabang, dan waktu pesan nggak punya hubungan sama sekali ke total penjualan.

In [7]:
# E. Prediksi Skenario Baru 
skenario = spark.createDataFrame(
    [
        ("Kopi Susu", "Magelang", "Pagi", 2, 12000),
        ("Roti Bakar", "Yogyakarta", "Malam", 4, 10000),
        ("Es Teh", "Semarang", "Siang", 5, 5000),
    ],
    ["nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan"]
)

skenario = indexer_menu.transform(skenario)
skenario = indexer_cabang.transform(skenario)
skenario = indexer_waktu.transform(skenario)
skenario = assembler.transform(skenario)

hasil = model.transform(skenario)
hasil.select("nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan", "prediction").show(truncate=False)

spark.stop()

+----------+----------+-----------+---+------------+------------------+
|nama_menu |cabang    |waktu_pesan|qty|harga_satuan|prediction        |
+----------+----------+-----------+---+------------+------------------+
|Kopi Susu |Magelang  |Pagi       |2  |12000       |13341.985039134117|
|Roti Bakar|Yogyakarta|Malam      |4  |10000       |51638.75039684451 |
|Es Teh    |Semarang  |Siang      |5  |5000        |59312.946902993426|
+----------+----------+-----------+---+------------+------------------+



Prediksi tiap skenario dihitung pakai indexer yang sama dengan data training, jadi urutan indeks kategorinya konsisten. Hasilnya lumayan deket sama hitungan manual (qty x harga), tapi masih ada selisih karena model linear nggak nangkep efek perkalian.